In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [10]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(text, rule="lemmatize"):
    """
    Clean and preprocess text data.

    Args:
        text (str): input sentence
        rule (str): "lemmatize" or "stem"

    Returns:
        str: cleaned text
    """

    # Lowercase
    data = text.lower()

    # Remove URLs
    data = re.sub(r"http\S+|www\S+", "", data)

    # Remove emojis
    data = re.sub(r"[^\w\s,]", "", data)

    # Remove punctuation & numbers
    data = data.translate(str.maketrans("", "", string.punctuation))
    data = re.sub(r"\d+", "", data)

    # Remove extra spaces
    data = re.sub(r"\s+", " ", data).strip()

    # Tokenization
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization or Stemming
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]
    else:
        raise ValueError("Pick between 'lemmatize' or 'stem'")

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [25]:
# Task 1: Loading dataaset
df = pd.read_csv("/content/drive/MyDrive/AI & ML/week8/trum_tweet_sentiment_analysis.csv")
df.columns

df.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [27]:
df["clean_text"] = df["text"].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

In [34]:
print(df.columns)

Index(['text', 'label', 'clean_text'], dtype='object')


In [33]:
df = df.rename(columns={"Sentiment": "label"})

train test split

In [35]:
from sklearn.model_selection import train_test_split

# Features and labels
X = df["clean_text"]
y = df["label"]

# Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   # IMPORTANT for classification
)

In [36]:
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nLabel distribution (train):")
print(y_train.value_counts(normalize=True))

print("\nLabel distribution (test):")
print(y_test.value_counts(normalize=True))

Train size: (1480098,)
Test size: (370025,)

Label distribution (train):
label
0    0.672502
1    0.327498
Name: proportion, dtype: float64

Label distribution (test):
label
0    0.672501
1    0.327499
Name: proportion, dtype: float64


TF-IDF Vectorization

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,   # limits vocabulary size (important for speed)
    ngram_range=(1, 2)   # unigrams + bigrams
)

# Fit ONLY on training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data (NO fitting)
X_test_tfidf = tfidf.transform(X_test)

In [38]:
print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

Train shape: (1480098, 5000)
Test shape: (370025, 5000)


Model Training and Evaluation

In [39]:
from sklearn.linear_model import LogisticRegression

# Initialize model
model = LogisticRegression(max_iter=1000)

# Train on TF-IDF features
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [40]:
y_pred = model.predict(X_test_tfidf)

In [41]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.94      0.93    248842
           1       0.88      0.82      0.84    121183

    accuracy                           0.90    370025
   macro avg       0.89      0.88      0.89    370025
weighted avg       0.90      0.90      0.90    370025

